<a href="https://colab.research.google.com/github/INDHUJA007-HUB/indhuja-day15-workshop/blob/main/Day7/MiniProject_7ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromaDB sentence-transformers -q
print("SUCCESS!!! GO AHEAD")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [ ]:
# standard libraries

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb

print("All libraries imported successfully")
print("The chromaDB version",chromadb.__version__)

All libraries imported successfully
The chromaDB version 1.5.9


In [ ]:
df=pd.DataFrame({
    "noteid":["1","2","3","4","5"],
    "subject":['Tamil','English','Maths','Science','Social'],
    'topic':['ETL','SQL Databases','Data Cleaning','APIs and Data Collection', 'Machine Learning Basics'],
    'content':[
        'ETL integrates data from various sources: extract, transform, load.',
        'SQL manages data in relational databases; used for inserting, searching, updating, deleting records.',
        'Data cleaning detects and corrects corrupt or inaccurate records in datasets.',
        'APIs allow applications to communicate, commonly used for programmatic data collection.',
        'Machine learning enables systems to learn from data to make decisions; includes supervised, unsupervised, and reinforcement learning.'
    ]
})

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Changed query_keyword to a more specific example from the content
# For example, let's query with the content of the first note (ETL)
query_keyword = df['content'].iloc[0] # Using an example from your dataset
model=SentenceTransformer("all-MiniLM-L6-v2")

# Encode the 'content' column of the DataFrame
df_encoding = model.encode(df['content'].tolist())
# Encode the query keyword
query_encoding = model.encode(query_keyword)

print("="*60)
print("Semantic Search query:", query_keyword)
print("="*60)

# Calculate cosine similarity between the query and all document embeddings
similarity_scores = cosine_similarity(query_encoding.reshape(1, -1), df_encoding)[0]
threshold = 0.4

# Iterate through the similarity scores and print results
for i, score in enumerate(similarity_scores):
  if score > threshold:
    print(f'FOUND [doc_id:{df["noteid"].iloc[i]}] (score={score:.3f}): {df["content"].iloc[i]}')
  else:
    print(f'NOT FOUND [doc_id:{df["noteid"].iloc[i]}] (score={score:.3f}): {df["content"].iloc[i]}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic Search query: ETL integrates data from various sources: extract, transform, load.
FOUND [doc_id:1] (score=1.000): ETL integrates data from various sources: extract, transform, load.
NOT FOUND [doc_id:2] (score=0.308): SQL manages data in relational databases; used for inserting, searching, updating, deleting records.
NOT FOUND [doc_id:3] (score=0.249): Data cleaning detects and corrects corrupt or inaccurate records in datasets.
FOUND [doc_id:4] (score=0.429): APIs allow applications to communicate, commonly used for programmatic data collection.
NOT FOUND [doc_id:5] (score=0.197): Machine learning enables systems to learn from data to make decisions; includes supervised, unsupervised, and reinforcement learning.


In [ ]:
chroma_client =chromadb.Client()
collection=chroma_client.get_or_create_collection("demo_notes")
print("ChromaDB client created (in-memory mode)")


print(f"Collection name:demo_notes ")
print(f"Documents in collection :{collection.count()}")

ChromaDB client created (in-memory mode)
Collection name:demo_notes 
Documents in collection :0


In [ ]:
# Prepare data for ChromaDB
ids = df['noteid'].tolist()
documents = df['content'].tolist()
metadatas = df[['subject', 'topic']].to_dict(orient='records')

# Add the documents to the collection
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully added {collection.count()} documents to the 'demo_notes' collection.")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 81.1MiB/s]


Successfully added 5 documents to the 'demo_notes' collection.
